In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"

SEQUENCE_DIR = STAGE1_DIR / "sequence_tensors"
OUTPUT_DIR = STAGE1_DIR / "experiments" / "sequence_model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = SEQUENCE_DIR / "sequence_tensor_summary.csv"
GRID_PATH = OUTPUT_DIR / "pre_sleep_sequence_experiment_grid.csv"

summary_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")

required_windows = [3, 5, 7]
required_splits = ["train", "validation", "test"]

for window in required_windows:
    window_summary = summary_df[summary_df["window"] == window]
    missing_splits = sorted(set(required_splits) - set(window_summary["split"]))
    if missing_splits:
        raise ValueError(f"window {window} missing splits: {missing_splits}")

candidate_configs = [
    {
        "model_family": "gru",
        "window": 3,
        "hidden_dim": 16,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
    {
        "model_family": "gru",
        "window": 5,
        "hidden_dim": 16,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
    {
        "model_family": "gru",
        "window": 7,
        "hidden_dim": 16,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
    {
        "model_family": "lstm",
        "window": 5,
        "hidden_dim": 16,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
    {
        "model_family": "bilstm",
        "window": 5,
        "hidden_dim": 12,
        "num_layers": 1,
        "bidirectional": True,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
    {
        "model_family": "cnn1d",
        "window": 5,
        "hidden_dim": 16,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.40,
        "weight_decay": 0.001,
        "learning_rate": 0.0008,
    },
]

rows = []

for idx, config in enumerate(candidate_configs):
    window = config["window"]

    split_paths = {
        split_name: SEQUENCE_DIR / f"window_{window}" / f"{split_name}.npz"
        for split_name in required_splits
    }

    for split_name, path in split_paths.items():
        if not path.exists():
            raise FileNotFoundError(path)

    train_row = summary_df[(summary_df["window"] == window) & (summary_df["split"] == "train")].iloc[0]
    val_row = summary_df[(summary_df["window"] == window) & (summary_df["split"] == "validation")].iloc[0]
    test_row = summary_df[(summary_df["window"] == window) & (summary_df["split"] == "test")].iloc[0]

    rows.append(
        {
            "experiment_id": f"presleep_seq_{idx:03d}",
            "objective": "strict_pre_sleep_forecasting",
            "feature_set": "design_c_stage1_sequence_58_features",
            "model_family": config["model_family"],
            "window": window,
            "input_dim": 58,
            "hidden_dim": config["hidden_dim"],
            "num_layers": config["num_layers"],
            "bidirectional": config["bidirectional"],
            "dropout": config["dropout"],
            "weight_decay": config["weight_decay"],
            "learning_rate": config["learning_rate"],
            "batch_size": 32,
            "max_epochs": 80,
            "patience": 12,
            "selection_metric": "validation_balanced_accuracy",
            "threshold_policy": "validation_balanced_accuracy",
            "train_samples": int(train_row["samples"]),
            "validation_samples": int(val_row["samples"]),
            "test_samples": int(test_row["samples"]),
            "train_path": str(split_paths["train"].relative_to(PROJECT_ROOT)),
            "validation_path": str(split_paths["validation"].relative_to(PROJECT_ROOT)),
            "test_path": str(split_paths["test"].relative_to(PROJECT_ROOT)),
        }
    )

grid_df = pd.DataFrame(rows)
grid_df.to_csv(GRID_PATH, index=False, encoding="utf-8-sig")

print("grid written:", GRID_PATH)
grid_df

grid written: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\pre_sleep_sequence_experiment_grid.csv


,experiment_id,objective,feature_set,model_family,window,input_dim,hidden_dim,num_layers,bidirectional,dropout,...,max_epochs,patience,selection_metric,threshold_policy,train_samples,validation_samples,test_samples,train_path,validation_path,test_path
0,presleep_seq_000,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,gru,3,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2234,329,853,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...
1,presleep_seq_001,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,gru,5,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2153,312,825,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...
2,presleep_seq_002,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,gru,7,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2073,296,797,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...
3,presleep_seq_003,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,lstm,5,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2153,312,825,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...
4,presleep_seq_004,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,bilstm,5,58,12,1,True,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2153,312,825,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...
5,presleep_seq_005,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,cnn1d,5,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2153,312,825,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"
GRID_PATH = STAGE1_DIR / "experiments" / "sequence_model_outputs" / "pre_sleep_sequence_experiment_grid.csv"

grid_df = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

required_npz_keys = {
    "X",
    "y",
    "participant_object_id",
    "target_sleep_start_datetime",
    "feature_names",
}

validation_rows = []

for _, row in grid_df.iterrows():
    experiment_id = row["experiment_id"]
    expected_window = int(row["window"])
    expected_features = int(row["input_dim"])

    for split_name, path_col, expected_samples_col in [
        ("train", "train_path", "train_samples"),
        ("validation", "validation_path", "validation_samples"),
        ("test", "test_path", "test_samples"),
    ]:
        npz_path = PROJECT_ROOT / row[path_col]
        expected_samples = int(row[expected_samples_col])

        if not npz_path.exists():
            raise FileNotFoundError(npz_path)

        with np.load(npz_path, allow_pickle=True) as arrays:
            keys = set(arrays.files)
            missing_keys = sorted(required_npz_keys - keys)
            if missing_keys:
                raise ValueError(f"{npz_path} missing keys: {missing_keys}")

            X = arrays["X"]
            y = arrays["y"]
            participants = arrays["participant_object_id"]
            feature_names = arrays["feature_names"]

            if X.ndim != 3:
                raise ValueError(f"{npz_path} X must be 3D, got {X.shape}")

            if X.shape[0] != expected_samples:
                raise ValueError(
                    f"{npz_path} sample mismatch: expected {expected_samples}, got {X.shape[0]}"
                )

            if X.shape[1] != expected_window:
                raise ValueError(
                    f"{npz_path} window mismatch: expected {expected_window}, got {X.shape[1]}"
                )

            if X.shape[2] != expected_features:
                raise ValueError(
                    f"{npz_path} feature mismatch: expected {expected_features}, got {X.shape[2]}"
                )

            if len(y) != X.shape[0]:
                raise ValueError(f"{npz_path} y length mismatch")

            if len(participants) != X.shape[0]:
                raise ValueError(f"{npz_path} participant length mismatch")

            if len(feature_names) != expected_features:
                raise ValueError(f"{npz_path} feature_names length mismatch")

            if not np.isfinite(X).all():
                raise ValueError(f"{npz_path} contains non-finite X values")

            validation_rows.append(
                {
                    "experiment_id": experiment_id,
                    "split": split_name,
                    "path": str(npz_path.relative_to(PROJECT_ROOT)),
                    "X_shape": str(X.shape),
                    "y_shape": str(y.shape),
                    "participants": int(pd.Series(participants).nunique()),
                    "positive_rate": float(np.mean(y)),
                }
            )

validation_df = pd.DataFrame(validation_rows)
validation_df

VALIDATION_PATH = STAGE1_DIR / "experiments" / "sequence_model_outputs" / "pre_sleep_sequence_grid_validation.csv"
validation_df.to_csv(VALIDATION_PATH, index=False, encoding="utf-8-sig")
print("validation written:", VALIDATION_PATH)

,experiment_id,split,path,X_shape,y_shape,participants,positive_rate
0,presleep_seq_000,train,data\processed\pre_sleep_forecasting\design_c_...,"(2234, 3, 58)","(2234,)",41,0.413608
1,presleep_seq_000,validation,data\processed\pre_sleep_forecasting\design_c_...,"(329, 3, 58)","(329,)",9,0.358663
2,presleep_seq_000,test,data\processed\pre_sleep_forecasting\design_c_...,"(853, 3, 58)","(853,)",14,0.364596
3,presleep_seq_001,train,data\processed\pre_sleep_forecasting\design_c_...,"(2153, 5, 58)","(2153,)",40,0.414306
4,presleep_seq_001,validation,data\processed\pre_sleep_forecasting\design_c_...,"(312, 5, 58)","(312,)",8,0.365385
5,presleep_seq_001,test,data\processed\pre_sleep_forecasting\design_c_...,"(825, 5, 58)","(825,)",14,0.360000
6,presleep_seq_002,train,data\processed\pre_sleep_forecasting\design_c_...,"(2073, 7, 58)","(2073,)",39,0.417752
7,presleep_seq_002,validation,data\processed\pre_sleep_forecasting\design_c_...,"(296, 7, 58)","(296,)",8,0.358108
8,presleep_seq_002,test,data\processed\pre_sleep_forecasting\design_c_...,"(797, 7, 58)","(797,)",14,0.360100
9,presleep_seq_003,train,data\processed\pre_sleep_forecasting\design_c_...,"(2153, 5, 58)","(2153,)",40,0.414306


In [5]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    brier_score_loss,
)


# =========================
# 1. Settings
# =========================

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"
EXPERIMENT_DIR = STAGE1_DIR / "experiments" / "sequence_model_outputs"
MODEL_DIR = EXPERIMENT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

GRID_PATH = EXPERIMENT_DIR / "pre_sleep_sequence_experiment_grid.csv"

EXPERIMENT_ID = "presleep_seq_000"
SEED = 2027

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)


# =========================
# 2. Reproducibility
# =========================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# =========================
# 3. Load Experiment Config
# =========================

grid_df = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

if EXPERIMENT_ID not in set(grid_df["experiment_id"]):
    raise ValueError(f"Unknown EXPERIMENT_ID: {EXPERIMENT_ID}")

config = grid_df.loc[grid_df["experiment_id"] == EXPERIMENT_ID].iloc[0].to_dict()

print("Selected experiment:")
display(pd.DataFrame([config]))


# =========================
# 4. Load NPZ Data
# =========================

def load_npz(relative_path):
    path = PROJECT_ROOT / relative_path
    arrays = np.load(path, allow_pickle=True)
    return {
        "X": arrays["X"].astype(np.float32),
        "y": arrays["y"].astype(np.float32),
        "participant_object_id": arrays["participant_object_id"],
        "target_sleep_start_datetime": arrays["target_sleep_start_datetime"],
        "feature_names": arrays["feature_names"],
        "path": path,
    }


train_data = load_npz(config["train_path"])
val_data = load_npz(config["validation_path"])
test_data = load_npz(config["test_path"])

print("Train X:", train_data["X"].shape, "y:", train_data["y"].shape)
print("Val   X:", val_data["X"].shape, "y:", val_data["y"].shape)
print("Test  X:", test_data["X"].shape, "y:", test_data["y"].shape)

assert train_data["X"].ndim == 3
assert val_data["X"].ndim == 3
assert test_data["X"].ndim == 3
assert train_data["X"].shape[2] == int(config["input_dim"])
assert len(train_data["feature_names"]) == int(config["input_dim"])


# =========================
# 5. Dataset / DataLoader
# =========================

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


batch_size = int(config["batch_size"])

train_loader = DataLoader(
    SequenceDataset(train_data["X"], train_data["y"]),
    batch_size=batch_size,
    shuffle=True,
)

val_loader = DataLoader(
    SequenceDataset(val_data["X"], val_data["y"]),
    batch_size=batch_size,
    shuffle=False,
)

test_loader = DataLoader(
    SequenceDataset(test_data["X"], test_data["y"]),
    batch_size=batch_size,
    shuffle=False,
)


# =========================
# 6. Models
# =========================

class RNNClassifier(nn.Module):
    def __init__(
        self,
        model_family: str,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool = False,
    ):
        super().__init__()

        self.model_family = model_family
        self.bidirectional = bidirectional

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if model_family in {"gru", "bilstm"}:
            if model_family == "gru":
                self.rnn = nn.GRU(
                    input_size=input_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    dropout=rnn_dropout,
                    bidirectional=bidirectional,
                )
            else:
                self.rnn = nn.LSTM(
                    input_size=input_dim,
                    hidden_size=hidden_dim,
                    num_layers=num_layers,
                    batch_first=True,
                    dropout=rnn_dropout,
                    bidirectional=True,
                )
        elif model_family == "lstm":
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=False,
            )
        else:
            raise ValueError(f"Unsupported RNN model_family: {model_family}")

        output_dim = hidden_dim * (2 if (bidirectional or model_family == "bilstm") else 1)

        self.head = nn.Sequential(
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
            nn.Linear(output_dim, 1),
        )

    def forward(self, x):
        output, _ = self.rnn(x)
        last_hidden = output[:, -1, :]
        return self.head(last_hidden).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # x: batch x time x features -> batch x features x time
        x = x.transpose(1, 2)
        x = self.network(x)
        return self.head(x).squeeze(-1)


def build_model(config):
    model_family = config["model_family"]
    input_dim = int(config["input_dim"])
    hidden_dim = int(config["hidden_dim"])
    num_layers = int(config["num_layers"])
    dropout = float(config["dropout"])
    bidirectional = bool(config["bidirectional"])

    if model_family in {"gru", "lstm", "bilstm"}:
        return RNNClassifier(
            model_family=model_family,
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=bidirectional,
        )

    if model_family == "cnn1d":
        return CNN1DClassifier(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

    raise ValueError(f"Unsupported model_family: {model_family}")


model = build_model(config).to(DEVICE)
print(model)


# =========================
# 7. Metrics Helpers
# =========================

def predict_loader(model, loader):
    model.eval()

    all_prob = []
    all_y = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            logits = model(X_batch)
            prob = torch.sigmoid(logits).detach().cpu().numpy()

            all_prob.append(prob)
            all_y.append(y_batch.numpy())

    return np.concatenate(all_y), np.concatenate(all_prob)


def find_best_threshold(y_true, probabilities, metric="balanced_accuracy"):
    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)

        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": balanced_accuracy_score(y_true, pred),
                "f1": f1_score(y_true, pred, zero_division=0),
                "precision": precision_score(y_true, pred, zero_division=0),
                "recall": recall_score(y_true, pred, zero_division=0),
            }
        )

    threshold_df = pd.DataFrame(rows)

    if metric == "balanced_accuracy":
        best_idx = threshold_df["balanced_accuracy"].idxmax()
    elif metric == "f1":
        best_idx = threshold_df["f1"].idxmax()
    else:
        raise ValueError(metric)

    return float(threshold_df.loc[best_idx, "threshold"]), threshold_df


def compute_metrics(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positive_rate": float(np.mean(y_true)),
        "mean_probability": float(np.mean(probabilities)),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, probabilities)
        metrics["average_precision"] = average_precision_score(y_true, probabilities)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics


# =========================
# 8. Training Setup
# =========================

positive_count = float(train_data["y"].sum())
negative_count = float(len(train_data["y"]) - positive_count)

pos_weight_value = negative_count / max(positive_count, 1.0)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(config["learning_rate"]),
    weight_decay=float(config["weight_decay"]),
)

max_epochs = int(config["max_epochs"])
patience = int(config["patience"])

print("positive_count:", positive_count)
print("negative_count:", negative_count)
print("pos_weight:", pos_weight_value)


# =========================
# 9. Train Loop
# =========================

best_val_balanced_accuracy = -np.inf
best_epoch = None
best_state = None
patience_counter = 0

history_rows = []

for epoch in range(1, max_epochs + 1):
    model.train()

    train_losses = []

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_losses.append(float(loss.detach().cpu().item()))

    train_y, train_prob = predict_loader(model, train_loader)
    val_y, val_prob = predict_loader(model, val_loader)

    selected_threshold, val_threshold_df = find_best_threshold(
        val_y,
        val_prob,
        metric="balanced_accuracy",
    )

    train_metrics = compute_metrics(train_y, train_prob, selected_threshold)
    val_metrics = compute_metrics(val_y, val_prob, selected_threshold)

    row = {
        "experiment_id": EXPERIMENT_ID,
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "selected_threshold": selected_threshold,
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_roc_auc": train_metrics["roc_auc"],
        "train_average_precision": train_metrics["average_precision"],
        "train_f1": train_metrics["f1"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "validation_balanced_accuracy": val_metrics["balanced_accuracy"],
        "validation_roc_auc": val_metrics["roc_auc"],
        "validation_average_precision": val_metrics["average_precision"],
        "validation_f1": val_metrics["f1"],
        "validation_precision": val_metrics["precision"],
        "validation_recall": val_metrics["recall"],
    }

    history_rows.append(row)

    improved = val_metrics["balanced_accuracy"] > best_val_balanced_accuracy

    if improved:
        best_val_balanced_accuracy = val_metrics["balanced_accuracy"]
        best_epoch = epoch
        best_state = {
            "model_state_dict": model.state_dict(),
            "config": config,
            "epoch": epoch,
            "selected_threshold": selected_threshold,
            "validation_metrics": val_metrics,
            "seed": SEED,
        }
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"epoch={epoch:03d} "
        f"loss={np.mean(train_losses):.4f} "
        f"val_ba={val_metrics['balanced_accuracy']:.4f} "
        f"threshold={selected_threshold:.2f} "
        f"patience={patience_counter}/{patience}"
    )

    if patience_counter >= patience:
        print("early stopping")
        break


history_df = pd.DataFrame(history_rows)

print("best_epoch:", best_epoch)
print("best_val_balanced_accuracy:", best_val_balanced_accuracy)


# =========================
# 10. Load Best State And Evaluate
# =========================

model.load_state_dict(best_state["model_state_dict"])
model.eval()

selected_threshold = float(best_state["selected_threshold"])

train_y, train_prob = predict_loader(model, train_loader)
val_y, val_prob = predict_loader(model, val_loader)
test_y, test_prob = predict_loader(model, test_loader)

train_metrics = compute_metrics(train_y, train_prob, selected_threshold)
val_metrics = compute_metrics(val_y, val_prob, selected_threshold)
test_metrics = compute_metrics(test_y, test_prob, selected_threshold)

metrics_rows = []

for split_name, metrics in [
    ("train", train_metrics),
    ("validation", val_metrics),
    ("test", test_metrics),
]:
    row = {
        "experiment_id": EXPERIMENT_ID,
        "split": split_name,
        "model_family": config["model_family"],
        "window": int(config["window"]),
        "input_dim": int(config["input_dim"]),
        "hidden_dim": int(config["hidden_dim"]),
        "dropout": float(config["dropout"]),
        "weight_decay": float(config["weight_decay"]),
        "learning_rate": float(config["learning_rate"]),
        "best_epoch": int(best_epoch),
        "seed": int(SEED),
        **metrics,
    }
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)

display(metrics_df)


# =========================
# 11. Predictions Table
# =========================

def make_prediction_df(split_name, data, y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    return pd.DataFrame(
        {
            "experiment_id": EXPERIMENT_ID,
            "split": split_name,
            "participant_object_id": data["participant_object_id"],
            "target_sleep_start_datetime": data["target_sleep_start_datetime"],
            "y_true": y_true.astype(int),
            "good_sleep_probability": probabilities,
            "good_sleep_pred": pred,
            "threshold": float(threshold),
        }
    )


prediction_df = pd.concat(
    [
        make_prediction_df("train", train_data, train_y, train_prob, selected_threshold),
        make_prediction_df("validation", val_data, val_y, val_prob, selected_threshold),
        make_prediction_df("test", test_data, test_y, test_prob, selected_threshold),
    ],
    ignore_index=True,
)

display(prediction_df.head())


# =========================
# 12. Save Outputs
# =========================

history_path = EXPERIMENT_DIR / f"{EXPERIMENT_ID}_training_history.csv"
metrics_path = EXPERIMENT_DIR / f"{EXPERIMENT_ID}_metrics.csv"
predictions_path = EXPERIMENT_DIR / f"{EXPERIMENT_ID}_predictions.csv"
threshold_path = EXPERIMENT_DIR / f"{EXPERIMENT_ID}_validation_threshold_sensitivity.csv"
model_path = MODEL_DIR / f"{EXPERIMENT_ID}_best.pt"
config_path = EXPERIMENT_DIR / f"{EXPERIMENT_ID}_config.json"

history_df.to_csv(history_path, index=False, encoding="utf-8-sig")
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
prediction_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")
val_threshold_df.to_csv(threshold_path, index=False, encoding="utf-8-sig")

torch.save(
    {
        "experiment_id": EXPERIMENT_ID,
        "model_family": config["model_family"],
        "input_dim": int(config["input_dim"]),
        "hidden_dim": int(config["hidden_dim"]),
        "num_layers": int(config["num_layers"]),
        "bidirectional": bool(config["bidirectional"]),
        "dropout": float(config["dropout"]),
        "window": int(config["window"]),
        "state_dict": best_state["model_state_dict"],
        "best_epoch": int(best_epoch),
        "selected_threshold": selected_threshold,
        "seed": int(SEED),
        "config": config,
        "feature_names": train_data["feature_names"],
    },
    model_path,
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment_id": EXPERIMENT_ID,
            "seed": SEED,
            "device": str(DEVICE),
            "config": {
                key: (
                    value.item()
                    if hasattr(value, "item")
                    else value
                )
                for key, value in config.items()
            },
            "best_epoch": int(best_epoch),
            "selected_threshold": selected_threshold,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("saved history:", history_path)
print("saved metrics:", metrics_path)
print("saved predictions:", predictions_path)
print("saved threshold sensitivity:", threshold_path)
print("saved model:", model_path)
print("saved config:", config_path)

DEVICE: cpu
Selected experiment:


,experiment_id,objective,feature_set,model_family,window,input_dim,hidden_dim,num_layers,bidirectional,dropout,...,max_epochs,patience,selection_metric,threshold_policy,train_samples,validation_samples,test_samples,train_path,validation_path,test_path
0,presleep_seq_000,strict_pre_sleep_forecasting,design_c_stage1_sequence_58_features,gru,3,58,16,1,False,0.4,...,80,12,validation_balanced_accuracy,validation_balanced_accuracy,2234,329,853,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...,data\processed\pre_sleep_forecasting\design_c_...


Train X: (2234, 3, 58) y: (2234,)
Val   X: (329, 3, 58) y: (329,)
Test  X: (853, 3, 58) y: (853,)
RNNClassifier(
  (rnn): GRU(58, 16, batch_first=True)
  (head): Sequential(
    (0): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.4, inplace=False)
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)
positive_count: 924.0
negative_count: 1310.0
pos_weight: 1.4177489177489178
epoch=001 loss=0.8175 val_ba=0.6854 threshold=0.41 patience=0/12
epoch=002 loss=0.7841 val_ba=0.6719 threshold=0.36 patience=1/12
epoch=003 loss=0.7739 val_ba=0.6601 threshold=0.35 patience=2/12
epoch=004 loss=0.7663 val_ba=0.6558 threshold=0.35 patience=3/12
epoch=005 loss=0.7534 val_ba=0.6539 threshold=0.40 patience=4/12
epoch=006 loss=0.7466 val_ba=0.6534 threshold=0.35 patience=5/12
epoch=007 loss=0.7384 val_ba=0.6576 threshold=0.40 patience=6/12
epoch=008 loss=0.7388 val_ba=0.6526 threshold=0.36 patience=7/12
epoch=009 loss=0.7356 val_ba=0.6517 threshold=0.36 pa

,experiment_id,split,model_family,window,input_dim,hidden_dim,dropout,weight_decay,learning_rate,best_epoch,...,recall,brier_score,tn,fp,fn,tp,positive_rate,mean_probability,roc_auc,average_precision
0,presleep_seq_000,train,gru,3,58,16,0.4,0.001,0.0008,1,...,0.799784,0.202775,725,585,185,739,0.413608,0.484784,0.753729,0.676726
1,presleep_seq_000,validation,gru,3,58,16,0.4,0.001,0.0008,1,...,0.711864,0.218376,116,95,34,84,0.358663,0.473036,0.706241,0.546252
2,presleep_seq_000,test,gru,3,58,16,0.4,0.001,0.0008,1,...,0.700965,0.210354,321,221,93,218,0.364596,0.434464,0.693466,0.599649


,experiment_id,split,participant_object_id,target_sleep_start_datetime,y_true,good_sleep_probability,good_sleep_pred,threshold
0,presleep_seq_000,train,621e2e8e67b776a24055b564,2021-05-25 23:46:30,0,0.328053,0,0.41
1,presleep_seq_000,train,621e2e8e67b776a24055b564,2021-05-26 23:21:30,0,0.471676,1,0.41
2,presleep_seq_000,train,621e2e8e67b776a24055b564,2021-05-27 23:37:00,1,0.275045,0,0.41
3,presleep_seq_000,train,621e2e8e67b776a24055b564,2021-05-28 23:50:00,0,0.563489,1,0.41
4,presleep_seq_000,train,621e2e8e67b776a24055b564,2021-05-30 00:30:00,1,0.286601,0,0.41


saved history: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\presleep_seq_000_training_history.csv
saved metrics: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\presleep_seq_000_metrics.csv
saved predictions: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\presleep_seq_000_predictions.csv
saved threshold sensitivity: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\presleep_seq_000_validation_threshold_sensitivity.csv
saved model: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_outputs\models\presleep_seq_000_best.pt
saved config: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\sequence_model_out

In [6]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    brier_score_loss,
)


# =========================
# 1. Settings
# =========================

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"
EXPERIMENT_DIR = STAGE1_DIR / "experiments" / "sequence_model_outputs"
MODEL_DIR = EXPERIMENT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

GRID_PATH = EXPERIMENT_DIR / "pre_sleep_sequence_experiment_grid.csv"

SEED = 2027
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OVERWRITE = False

print("DEVICE:", DEVICE)
print("OVERWRITE:", OVERWRITE)


# =========================
# 2. Reproducibility
# =========================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================
# 3. Data Helpers
# =========================

def load_npz(project_root: Path, relative_path):
    path = project_root / relative_path
    arrays = np.load(path, allow_pickle=True)
    return {
        "X": arrays["X"].astype(np.float32),
        "y": arrays["y"].astype(np.float32),
        "participant_object_id": arrays["participant_object_id"],
        "target_sleep_start_datetime": arrays["target_sleep_start_datetime"],
        "feature_names": arrays["feature_names"],
        "path": path,
    }


class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(data, batch_size, shuffle):
    return DataLoader(
        SequenceDataset(data["X"], data["y"]),
        batch_size=batch_size,
        shuffle=shuffle,
    )


# =========================
# 4. Models
# =========================

class RNNClassifier(nn.Module):
    def __init__(
        self,
        model_family: str,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool = False,
    ):
        super().__init__()

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if model_family == "gru":
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional,
            )
            output_dim = hidden_dim * (2 if bidirectional else 1)

        elif model_family == "lstm":
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=False,
            )
            output_dim = hidden_dim

        elif model_family == "bilstm":
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=True,
            )
            output_dim = hidden_dim * 2

        else:
            raise ValueError(f"Unsupported RNN model_family: {model_family}")

        self.head = nn.Sequential(
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
            nn.Linear(output_dim, 1),
        )

    def forward(self, x):
        output, _ = self.rnn(x)
        last_hidden = output[:, -1, :]
        return self.head(last_hidden).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.network(x)
        return self.head(x).squeeze(-1)


def build_model(config):
    model_family = config["model_family"]
    input_dim = int(config["input_dim"])
    hidden_dim = int(config["hidden_dim"])
    num_layers = int(config["num_layers"])
    dropout = float(config["dropout"])
    bidirectional = bool(config["bidirectional"])

    if model_family in {"gru", "lstm", "bilstm"}:
        return RNNClassifier(
            model_family=model_family,
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=bidirectional,
        )

    if model_family == "cnn1d":
        return CNN1DClassifier(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

    raise ValueError(f"Unsupported model_family: {model_family}")


# =========================
# 5. Metrics Helpers
# =========================

def predict_loader(model, loader):
    model.eval()

    all_prob = []
    all_y = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            logits = model(X_batch)
            prob = torch.sigmoid(logits).detach().cpu().numpy()

            all_prob.append(prob)
            all_y.append(y_batch.numpy())

    return np.concatenate(all_y), np.concatenate(all_prob)


def find_best_threshold(y_true, probabilities, metric="balanced_accuracy"):
    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)

        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": balanced_accuracy_score(y_true, pred),
                "f1": f1_score(y_true, pred, zero_division=0),
                "precision": precision_score(y_true, pred, zero_division=0),
                "recall": recall_score(y_true, pred, zero_division=0),
            }
        )

    threshold_df = pd.DataFrame(rows)

    if metric == "balanced_accuracy":
        best_idx = threshold_df["balanced_accuracy"].idxmax()
    elif metric == "f1":
        best_idx = threshold_df["f1"].idxmax()
    else:
        raise ValueError(metric)

    return float(threshold_df.loc[best_idx, "threshold"]), threshold_df


def compute_metrics(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positive_rate": float(np.mean(y_true)),
        "mean_probability": float(np.mean(probabilities)),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, probabilities)
        metrics["average_precision"] = average_precision_score(y_true, probabilities)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics


def make_prediction_df(experiment_id, split_name, data, y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    return pd.DataFrame(
        {
            "experiment_id": experiment_id,
            "split": split_name,
            "participant_object_id": data["participant_object_id"],
            "target_sleep_start_datetime": data["target_sleep_start_datetime"],
            "y_true": y_true.astype(int),
            "good_sleep_probability": probabilities,
            "good_sleep_pred": pred,
            "threshold": float(threshold),
        }
    )


# =========================
# 6. One Experiment Runner
# =========================

def run_experiment(config):
    experiment_id = config["experiment_id"]

    history_path = EXPERIMENT_DIR / f"{experiment_id}_training_history.csv"
    metrics_path = EXPERIMENT_DIR / f"{experiment_id}_metrics.csv"
    predictions_path = EXPERIMENT_DIR / f"{experiment_id}_predictions.csv"
    threshold_path = EXPERIMENT_DIR / f"{experiment_id}_validation_threshold_sensitivity.csv"
    model_path = MODEL_DIR / f"{experiment_id}_best.pt"
    config_path = EXPERIMENT_DIR / f"{experiment_id}_config.json"

    output_paths = [
        history_path,
        metrics_path,
        predictions_path,
        threshold_path,
        model_path,
        config_path,
    ]

    if not OVERWRITE and all(path.exists() for path in output_paths):
        print(f"[SKIP] {experiment_id}: outputs already exist")
        metrics_df = pd.read_csv(metrics_path, encoding="utf-8-sig")
        return metrics_df

    set_seed(SEED)

    train_data = load_npz(PROJECT_ROOT, config["train_path"])
    val_data = load_npz(PROJECT_ROOT, config["validation_path"])
    test_data = load_npz(PROJECT_ROOT, config["test_path"])

    assert train_data["X"].ndim == 3
    assert val_data["X"].ndim == 3
    assert test_data["X"].ndim == 3
    assert train_data["X"].shape[2] == int(config["input_dim"])
    assert len(train_data["feature_names"]) == int(config["input_dim"])

    batch_size = int(config["batch_size"])

    train_loader = make_loader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = make_loader(test_data, batch_size=batch_size, shuffle=False)

    model = build_model(config).to(DEVICE)

    positive_count = float(train_data["y"].sum())
    negative_count = float(len(train_data["y"]) - positive_count)

    pos_weight_value = negative_count / max(positive_count, 1.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config["learning_rate"]),
        weight_decay=float(config["weight_decay"]),
    )

    max_epochs = int(config["max_epochs"])
    patience = int(config["patience"])

    best_val_balanced_accuracy = -np.inf
    best_epoch = None
    best_state = None
    best_threshold_df = None
    patience_counter = 0

    history_rows = []

    start_time = time.time()

    print(
        f"[RUN] {experiment_id} "
        f"model={config['model_family']} "
        f"window={config['window']} "
        f"train={len(train_data['y'])} "
        f"val={len(val_data['y'])} "
        f"test={len(test_data['y'])}"
    )

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        train_y, train_prob = predict_loader(model, train_loader)
        val_y, val_prob = predict_loader(model, val_loader)

        selected_threshold, val_threshold_df = find_best_threshold(
            val_y,
            val_prob,
            metric="balanced_accuracy",
        )

        train_metrics = compute_metrics(train_y, train_prob, selected_threshold)
        val_metrics = compute_metrics(val_y, val_prob, selected_threshold)

        history_rows.append(
            {
                "experiment_id": experiment_id,
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "selected_threshold": selected_threshold,
                "train_balanced_accuracy": train_metrics["balanced_accuracy"],
                "train_roc_auc": train_metrics["roc_auc"],
                "train_average_precision": train_metrics["average_precision"],
                "train_f1": train_metrics["f1"],
                "train_precision": train_metrics["precision"],
                "train_recall": train_metrics["recall"],
                "validation_balanced_accuracy": val_metrics["balanced_accuracy"],
                "validation_roc_auc": val_metrics["roc_auc"],
                "validation_average_precision": val_metrics["average_precision"],
                "validation_f1": val_metrics["f1"],
                "validation_precision": val_metrics["precision"],
                "validation_recall": val_metrics["recall"],
            }
        )

        improved = val_metrics["balanced_accuracy"] > best_val_balanced_accuracy

        if improved:
            best_val_balanced_accuracy = val_metrics["balanced_accuracy"]
            best_epoch = epoch
            best_state = {
                "model_state_dict": model.state_dict(),
                "config": config,
                "epoch": epoch,
                "selected_threshold": selected_threshold,
                "validation_metrics": val_metrics,
                "seed": SEED,
            }
            best_threshold_df = val_threshold_df.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"  epoch={epoch:03d} "
            f"loss={np.mean(train_losses):.4f} "
            f"val_ba={val_metrics['balanced_accuracy']:.4f} "
            f"threshold={selected_threshold:.2f} "
            f"patience={patience_counter}/{patience}"
        )

        if patience_counter >= patience:
            print("  early stopping")
            break

    if best_state is None:
        raise RuntimeError(f"No best state captured for {experiment_id}")

    model.load_state_dict(best_state["model_state_dict"])
    model.eval()

    selected_threshold = float(best_state["selected_threshold"])

    train_y, train_prob = predict_loader(model, train_loader)
    val_y, val_prob = predict_loader(model, val_loader)
    test_y, test_prob = predict_loader(model, test_loader)

    train_metrics = compute_metrics(train_y, train_prob, selected_threshold)
    val_metrics = compute_metrics(val_y, val_prob, selected_threshold)
    test_metrics = compute_metrics(test_y, test_prob, selected_threshold)

    metrics_rows = []

    for split_name, metrics in [
        ("train", train_metrics),
        ("validation", val_metrics),
        ("test", test_metrics),
    ]:
        metrics_rows.append(
            {
                "experiment_id": experiment_id,
                "split": split_name,
                "model_family": config["model_family"],
                "window": int(config["window"]),
                "input_dim": int(config["input_dim"]),
                "hidden_dim": int(config["hidden_dim"]),
                "dropout": float(config["dropout"]),
                "weight_decay": float(config["weight_decay"]),
                "learning_rate": float(config["learning_rate"]),
                "best_epoch": int(best_epoch),
                "seed": int(SEED),
                **metrics,
            }
        )

    history_df = pd.DataFrame(history_rows)
    metrics_df = pd.DataFrame(metrics_rows)

    prediction_df = pd.concat(
        [
            make_prediction_df(experiment_id, "train", train_data, train_y, train_prob, selected_threshold),
            make_prediction_df(experiment_id, "validation", val_data, val_y, val_prob, selected_threshold),
            make_prediction_df(experiment_id, "test", test_data, test_y, test_prob, selected_threshold),
        ],
        ignore_index=True,
    )

    history_df.to_csv(history_path, index=False, encoding="utf-8-sig")
    metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
    prediction_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")
    best_threshold_df.to_csv(threshold_path, index=False, encoding="utf-8-sig")

    torch.save(
        {
            "experiment_id": experiment_id,
            "model_family": config["model_family"],
            "input_dim": int(config["input_dim"]),
            "hidden_dim": int(config["hidden_dim"]),
            "num_layers": int(config["num_layers"]),
            "bidirectional": bool(config["bidirectional"]),
            "dropout": float(config["dropout"]),
            "window": int(config["window"]),
            "state_dict": best_state["model_state_dict"],
            "best_epoch": int(best_epoch),
            "selected_threshold": selected_threshold,
            "seed": int(SEED),
            "config": config,
            "feature_names": train_data["feature_names"],
        },
        model_path,
    )

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "experiment_id": experiment_id,
                "seed": SEED,
                "device": str(DEVICE),
                "config": {
                    key: (value.item() if hasattr(value, "item") else value)
                    for key, value in config.items()
                },
                "best_epoch": int(best_epoch),
                "selected_threshold": selected_threshold,
                "elapsed_seconds": float(time.time() - start_time),
            },
            f,
            indent=2,
            ensure_ascii=False,
        )

    test_row = metrics_df[metrics_df["split"] == "test"].iloc[0]
    print(
        f"[DONE] {experiment_id} "
        f"best_epoch={best_epoch} "
        f"threshold={selected_threshold:.2f} "
        f"test_ba={test_row['balanced_accuracy']:.4f} "
        f"test_auc={test_row['roc_auc']:.4f} "
        f"test_ap={test_row['average_precision']:.4f}"
    )

    return metrics_df


# =========================
# 7. Run Full Grid
# =========================

grid_df = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

all_metrics = []

for _, row in grid_df.iterrows():
    config = row.to_dict()
    metrics_df = run_experiment(config)
    all_metrics.append(metrics_df)

all_metrics_df = pd.concat(all_metrics, ignore_index=True)

combined_metrics_path = EXPERIMENT_DIR / "pre_sleep_sequence_model_metrics.csv"
all_metrics_df.to_csv(combined_metrics_path, index=False, encoding="utf-8-sig")

print("combined metrics written:", combined_metrics_path)

test_rank_df = (
    all_metrics_df[all_metrics_df["split"] == "test"]
    .sort_values("balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)

validation_rank_df = (
    all_metrics_df[all_metrics_df["split"] == "validation"]
    .sort_values("balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)

test_rank_path = EXPERIMENT_DIR / "pre_sleep_sequence_model_test_rank.csv"
validation_rank_path = EXPERIMENT_DIR / "pre_sleep_sequence_model_validation_rank.csv"

test_rank_df.to_csv(test_rank_path, index=False, encoding="utf-8-sig")
validation_rank_df.to_csv(validation_rank_path, index=False, encoding="utf-8-sig")

print("test rank written:", test_rank_path)
print("validation rank written:", validation_rank_path)

display(validation_rank_df)
display(test_rank_df)

DEVICE: cpu
OVERWRITE: False
[SKIP] presleep_seq_000: outputs already exist
[RUN] presleep_seq_001 model=gru window=5 train=2153 val=312 test=825
  epoch=001 loss=0.8077 val_ba=0.6822 threshold=0.36 patience=0/12
  epoch=002 loss=0.7883 val_ba=0.6721 threshold=0.37 patience=1/12
  epoch=003 loss=0.7608 val_ba=0.6544 threshold=0.35 patience=2/12
  epoch=004 loss=0.7442 val_ba=0.6570 threshold=0.36 patience=3/12
  epoch=005 loss=0.7463 val_ba=0.6563 threshold=0.35 patience=4/12
  epoch=006 loss=0.7479 val_ba=0.6595 threshold=0.35 patience=5/12
  epoch=007 loss=0.7463 val_ba=0.6519 threshold=0.35 patience=6/12
  epoch=008 loss=0.7251 val_ba=0.6512 threshold=0.34 patience=7/12
  epoch=009 loss=0.7289 val_ba=0.6477 threshold=0.36 patience=8/12
  epoch=010 loss=0.7206 val_ba=0.6515 threshold=0.36 patience=9/12
  epoch=011 loss=0.7194 val_ba=0.6566 threshold=0.37 patience=10/12
  epoch=012 loss=0.7082 val_ba=0.6641 threshold=0.36 patience=11/12
  epoch=013 loss=0.7102 val_ba=0.6692 threshold=

,experiment_id,split,model_family,window,input_dim,hidden_dim,dropout,weight_decay,learning_rate,best_epoch,...,recall,brier_score,tn,fp,fn,tp,positive_rate,mean_probability,roc_auc,average_precision
0,presleep_seq_001,validation,gru,5,58,16,0.4,0.001,0.0008,1,...,0.859649,0.221054,89,109,16,98,0.365385,0.486013,0.707780,0.549697
1,presleep_seq_002,validation,gru,7,58,16,0.4,0.001,0.0008,20,...,0.839623,0.212885,86,104,17,89,0.358108,0.462868,0.720705,0.575998
2,presleep_seq_004,validation,bilstm,5,58,12,0.4,0.001,0.0008,4,...,0.605263,0.230915,134,64,45,69,0.365385,0.469897,0.684211,0.503347
3,presleep_seq_003,validation,lstm,5,58,16,0.4,0.001,0.0008,22,...,0.745614,0.234711,105,93,29,85,0.365385,0.478064,0.704102,0.553182
4,presleep_seq_000,validation,gru,3,58,16,0.4,0.001,0.0008,1,...,0.711864,0.218376,116,95,34,84,0.358663,0.473036,0.706241,0.546252
5,presleep_seq_005,validation,cnn1d,5,58,16,0.4,0.001,0.0008,18,...,0.807018,0.212972,90,108,22,92,0.365385,0.455997,0.706539,0.609039


,experiment_id,split,model_family,window,input_dim,hidden_dim,dropout,weight_decay,learning_rate,best_epoch,...,recall,brier_score,tn,fp,fn,tp,positive_rate,mean_probability,roc_auc,average_precision
0,presleep_seq_000,test,gru,3,58,16,0.4,0.001,0.0008,1,...,0.700965,0.210354,321,221,93,218,0.364596,0.434464,0.693466,0.599649
1,presleep_seq_004,test,bilstm,5,58,12,0.4,0.001,0.0008,4,...,0.511785,0.217224,394,134,145,152,0.360000,0.445294,0.677769,0.572002
2,presleep_seq_001,test,gru,5,58,16,0.4,0.001,0.0008,1,...,0.794613,0.214093,225,303,61,236,0.360000,0.452291,0.691001,0.599954
3,presleep_seq_002,test,gru,7,58,16,0.4,0.001,0.0008,20,...,0.759582,0.217953,226,284,69,218,0.360100,0.436383,0.676710,0.571153
4,presleep_seq_005,test,cnn1d,5,58,16,0.4,0.001,0.0008,18,...,0.794613,0.213192,204,324,61,236,0.360000,0.437794,0.688763,0.614669
5,presleep_seq_003,test,lstm,5,58,16,0.4,0.001,0.0008,22,...,0.602694,0.240459,283,245,118,179,0.360000,0.419236,0.613356,0.521457


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"

HP_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_hyperparameter_stability_outputs"
CALIBRATION_OUTPUT_DIR = STAGE1_DIR / "experiments" / "calibration_followup_outputs"
CALIBRATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = HP_OUTPUT_DIR / "stage1_hyperparameter_stability_predictions.csv"
SELECTION_PATH = HP_OUTPUT_DIR / "stage1_best_config_seed_selection_summary.csv"

SELECTED_EXPERIMENT_ID = "presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027"

pred_df = pd.read_csv(PREDICTION_PATH, encoding="utf-8-sig")
selection_df = pd.read_csv(SELECTION_PATH, encoding="utf-8-sig")

print("prediction shape:", pred_df.shape)
print("selection shape:", selection_df.shape)
print("prediction columns:")
print(pred_df.columns.tolist())

selected_pred_df = pred_df[pred_df["experiment_id"] == SELECTED_EXPERIMENT_ID].copy()

print("selected prediction shape:", selected_pred_df.shape)
print("splits:")
print(selected_pred_df["split"].value_counts())

required_columns = [
    "experiment_id",
    "split",
    "participant_object_id",
    "y_true",
    "y_probability",
]

missing_columns = [col for col in required_columns if col not in selected_pred_df.columns]
if missing_columns:
    print("Missing expected columns:", missing_columns)
    print("Available columns:", selected_pred_df.columns.tolist())
else:
    print("All expected columns found.")

display(selected_pred_df.head())

prediction shape: (56816, 10)
selection shape: (4, 19)
prediction columns:
['experiment_id', 'config_id', 'split', 'seed', 'sleep_episode_id', 'participant_object_id', 'y_true', 'y_probability', 'selected_threshold_from_validation', 'y_pred']
selected prediction shape: (3551, 10)
splits:
split
train         2323
test           881
validation     347
Name: count, dtype: int64
All expected columns found.


,experiment_id,config_id,split,seed,sleep_episode_id,participant_object_id,y_true,y_probability,selected_threshold_from_validation,y_pred
39061,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,train,2027,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,1,0.423425,0.54,0
39062,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,train,2027,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,1,0.575802,0.54,1
39063,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,train,2027,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,1,0.608308,0.54,1
39064,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,train,2027,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,1,0.593617,0.54,1
39065,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,train,2027,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,1,0.618604,0.54,1


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    brier_score_loss,
)


# =========================
# 1. Settings
# =========================

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
STAGE1_DIR = PROJECT_ROOT / "data" / "processed" / "pre_sleep_forecasting" / "design_c_stage1"

HP_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_hyperparameter_stability_outputs"
CALIBRATION_OUTPUT_DIR = STAGE1_DIR / "experiments" / "calibration_followup_outputs"
CALIBRATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = HP_OUTPUT_DIR / "stage1_hyperparameter_stability_predictions.csv"

SELECTED_EXPERIMENT_ID = "presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027"
OFFICIAL_THRESHOLD = 0.54

METHOD_COMPARISON_PATH = CALIBRATION_OUTPUT_DIR / "selected_model_calibration_method_comparison.csv"
CALIBRATED_PREDICTION_PATH = CALIBRATION_OUTPUT_DIR / "selected_model_calibrated_predictions.csv"
CALIBRATION_TABLE_PATH = CALIBRATION_OUTPUT_DIR / "selected_model_calibrated_calibration_table.csv"
SUMMARY_MD_PATH = CALIBRATION_OUTPUT_DIR / "selected_model_calibration_followup_summary.md"


# =========================
# 2. Helpers
# =========================

def clip_probability(p, eps=1e-6):
    return np.clip(np.asarray(p, dtype=float), eps, 1 - eps)


def logit(p):
    p = clip_probability(p)
    return np.log(p / (1 - p))


def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(prob, bins[1:-1], right=False)

    rows = []
    ece = 0.0
    mce = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        count = int(mask.sum())

        if count == 0:
            rows.append(
                {
                    "bin_id": bin_id,
                    "bin_left": bins[bin_id],
                    "bin_right": bins[bin_id + 1],
                    "count": 0,
                    "mean_probability": np.nan,
                    "observed_positive_rate": np.nan,
                    "absolute_error": np.nan,
                }
            )
            continue

        mean_probability = float(prob[mask].mean())
        observed_positive_rate = float(y_true[mask].mean())
        absolute_error = abs(mean_probability - observed_positive_rate)

        weight = count / len(y_true)
        ece += weight * absolute_error
        mce = max(mce, absolute_error)

        rows.append(
            {
                "bin_id": bin_id,
                "bin_left": bins[bin_id],
                "bin_right": bins[bin_id + 1],
                "count": count,
                "mean_probability": mean_probability,
                "observed_positive_rate": observed_positive_rate,
                "absolute_error": absolute_error,
            }
        )

    return float(ece), float(mce), pd.DataFrame(rows)


def find_best_threshold(y_true, prob, metric="balanced_accuracy"):
    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []

    for threshold in thresholds:
        pred = (prob >= threshold).astype(int)

        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": balanced_accuracy_score(y_true, pred),
                "f1": f1_score(y_true, pred, zero_division=0),
                "precision": precision_score(y_true, pred, zero_division=0),
                "recall": recall_score(y_true, pred, zero_division=0),
            }
        )

    threshold_df = pd.DataFrame(rows)

    if metric == "balanced_accuracy":
        best_idx = threshold_df["balanced_accuracy"].idxmax()
    elif metric == "f1":
        best_idx = threshold_df["f1"].idxmax()
    else:
        raise ValueError(metric)

    return float(threshold_df.loc[best_idx, "threshold"]), threshold_df


def compute_metrics(y_true, prob, threshold, method_name, split):
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ece, mce, _ = expected_calibration_error(y_true, prob, n_bins=10)

    return {
        "method": method_name,
        "split": split,
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob),
        "average_precision": average_precision_score(y_true, prob),
        "f1": f1_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, prob),
        "expected_calibration_error": ece,
        "maximum_calibration_error": mce,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positive_rate": float(np.mean(y_true)),
        "mean_probability": float(np.mean(prob)),
    }


# =========================
# 3. Load Predictions
# =========================

pred_df = pd.read_csv(PREDICTION_PATH, encoding="utf-8-sig")
selected_df = pred_df[pred_df["experiment_id"] == SELECTED_EXPERIMENT_ID].copy()

val_df = selected_df[selected_df["split"] == "validation"].copy()
test_df = selected_df[selected_df["split"] == "test"].copy()

y_val = val_df["y_true"].to_numpy(dtype=int)
p_val = val_df["y_probability"].to_numpy(dtype=float)

y_test = test_df["y_true"].to_numpy(dtype=int)
p_test = test_df["y_probability"].to_numpy(dtype=float)

print("validation:", len(y_val), "positive_rate:", y_val.mean())
print("test:", len(y_test), "positive_rate:", y_test.mean())


# =========================
# 4. Fit Calibration Models On Validation
# =========================

# Original uncalibrated probability.
p_val_original = p_val.copy()
p_test_original = p_test.copy()

# Platt scaling: logistic regression on logit(probability).
platt = LogisticRegression(solver="lbfgs")
platt.fit(logit(p_val).reshape(-1, 1), y_val)

p_val_platt = platt.predict_proba(logit(p_val).reshape(-1, 1))[:, 1]
p_test_platt = platt.predict_proba(logit(p_test).reshape(-1, 1))[:, 1]

# Isotonic regression: monotonic nonparametric calibration.
isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(p_val, y_val)

p_val_isotonic = isotonic.predict(p_val)
p_test_isotonic = isotonic.predict(p_test)


# =========================
# 5. Threshold Selection On Validation
# =========================

method_probs = {
    "original": {
        "validation": p_val_original,
        "test": p_test_original,
    },
    "platt": {
        "validation": p_val_platt,
        "test": p_test_platt,
    },
    "isotonic": {
        "validation": p_val_isotonic,
        "test": p_test_isotonic,
    },
}

thresholds = {}

for method_name, probs in method_probs.items():
    if method_name == "original":
        # Keep official threshold as primary reference.
        thresholds[method_name] = OFFICIAL_THRESHOLD
    else:
        selected_threshold, _ = find_best_threshold(
            y_val,
            probs["validation"],
            metric="balanced_accuracy",
        )
        thresholds[method_name] = selected_threshold

print("thresholds:", thresholds)


# =========================
# 6. Metrics
# =========================

metrics_rows = []

for method_name, probs in method_probs.items():
    threshold = thresholds[method_name]

    metrics_rows.append(
        compute_metrics(
            y_val,
            probs["validation"],
            threshold,
            method_name,
            "validation",
        )
    )

    metrics_rows.append(
        compute_metrics(
            y_test,
            probs["test"],
            threshold,
            method_name,
            "test",
        )
    )

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METHOD_COMPARISON_PATH, index=False, encoding="utf-8-sig")

display(metrics_df)


# =========================
# 7. Calibrated Predictions
# =========================

calibrated_test_df = test_df.copy()
calibrated_test_df["prob_original"] = p_test_original
calibrated_test_df["prob_platt"] = p_test_platt
calibrated_test_df["prob_isotonic"] = p_test_isotonic

for method_name in ["original", "platt", "isotonic"]:
    prob_col = f"prob_{method_name}"
    pred_col = f"pred_{method_name}"
    threshold = thresholds[method_name]
    calibrated_test_df[pred_col] = (calibrated_test_df[prob_col] >= threshold).astype(int)
    calibrated_test_df[f"threshold_{method_name}"] = threshold

calibrated_test_df.to_csv(CALIBRATED_PREDICTION_PATH, index=False, encoding="utf-8-sig")


# =========================
# 8. Calibration Tables
# =========================

table_rows = []

for method_name, probs in method_probs.items():
    for split_name, y_true, prob in [
        ("validation", y_val, probs["validation"]),
        ("test", y_test, probs["test"]),
    ]:
        _, _, table_df = expected_calibration_error(y_true, prob, n_bins=10)
        table_df.insert(0, "method", method_name)
        table_df.insert(1, "split", split_name)
        table_rows.append(table_df)

calibration_table_df = pd.concat(table_rows, ignore_index=True)
calibration_table_df.to_csv(CALIBRATION_TABLE_PATH, index=False, encoding="utf-8-sig")


# =========================
# 9. Markdown Summary
# =========================

test_metrics = metrics_df[metrics_df["split"] == "test"].copy()
test_metrics = test_metrics.sort_values("brier_score", ascending=True)

best_brier_row = test_metrics.iloc[0]
best_ece_row = test_metrics.sort_values("expected_calibration_error", ascending=True).iloc[0]

summary_lines = [
    "# Selected Pre-Sleep Model Calibration Follow-Up Summary",
    "",
    f"- Selected experiment: `{SELECTED_EXPERIMENT_ID}`",
    f"- Calibration fitting split: `validation`",
    f"- Evaluation split: `test`",
    f"- Official original threshold: `{OFFICIAL_THRESHOLD}`",
    "",
    "## Test Metrics",
    "",
    test_metrics[
        [
            "method",
            "threshold",
            "balanced_accuracy",
            "roc_auc",
            "average_precision",
            "brier_score",
            "expected_calibration_error",
            "precision",
            "recall",
        ]
    ].to_markdown(index=False),
    "",
    "## Best Calibration By Metric",
    "",
    f"- Best Brier score: `{best_brier_row['method']}` = `{best_brier_row['brier_score']:.4f}`",
    f"- Best ECE: `{best_ece_row['method']}` = `{best_ece_row['expected_calibration_error']:.4f}`",
    "",
    "## Interpretation Guardrail",
    "",
    "Calibration correction is optional probability post-processing. The model remains research-grade and should not be used for clinical or high-stakes decisions.",
    "",
]

SUMMARY_MD_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

print("method comparison:", METHOD_COMPARISON_PATH)
print("calibrated predictions:", CALIBRATED_PREDICTION_PATH)
print("calibration table:", CALIBRATION_TABLE_PATH)
print("summary markdown:", SUMMARY_MD_PATH)

validation: 347 positive_rate: 0.3515850144092219
test: 881 positive_rate: 0.36095346197502837
thresholds: {'original': 0.54, 'platt': 0.4, 'isotonic': 0.34}


,method,split,threshold,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,brier_score,expected_calibration_error,maximum_calibration_error,tn,fp,fn,tp,positive_rate,mean_probability
0,original,validation,0.54,0.676958,0.682368,0.512725,0.584000,0.570312,0.598361,0.220914,1.177506e-01,2.826827e-01,170,55,49,73,0.351585,0.469336
1,original,test,0.54,0.649209,0.693695,0.618684,0.515267,0.655340,0.424528,0.212638,1.256314e-01,8.342340e-01,492,71,183,135,0.360953,0.439648
2,platt,validation,0.40,0.669945,0.682368,0.512725,0.578125,0.552239,0.606557,0.207017,5.618778e-02,1.698019e-01,165,60,48,74,0.351585,0.351597
3,platt,test,0.40,0.656459,0.693695,0.618684,0.531599,0.650000,0.449686,0.208325,7.448096e-02,9.136720e-01,486,77,175,143,0.360953,0.320599
4,isotonic,validation,0.34,0.676958,0.712295,0.532039,0.584000,0.570312,0.598361,0.195063,6.926896e-17,1.110223e-16,170,55,49,73,0.351585,0.351585
5,isotonic,test,0.34,0.649209,0.690154,0.558440,0.515267,0.655340,0.424528,0.208070,4.858056e-02,3.476190e-01,492,71,183,135,0.360953,0.316535


method comparison: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\calibration_followup_outputs\selected_model_calibration_method_comparison.csv
calibrated predictions: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\calibration_followup_outputs\selected_model_calibrated_predictions.csv
calibration table: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\calibration_followup_outputs\selected_model_calibrated_calibration_table.csv
summary markdown: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\calibration_followup_outputs\selected_model_calibration_followup_summary.md
